# 2-2 정형 피처 베이스라인 모델

`is_low_rating_surge` 예측 — 리뷰 텍스트 없이 정형 피처만 사용한 기준 모델.

## 1. 데이터 로드

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report

df = pd.read_parquet("../../data/processed/product_month_labeled.parquet")

## 2. 신뢰도 피처 추가 (Wilson score interval)

In [ ]:
from eda import add_reliability_features

df = add_reliability_features(df)  # reliability_ci_width 컬럼 추가

## 3. 최종 피처셋 확정

VIF/상관관계 검토 결과에 따라 6개만 사용 (`avg_rating`, `low_rating_count`, `text_available_ratio`, `past_3m_review_count`, `past_3m_low_rating_count`는 중복이라 제외).

In [ ]:
FEATURE_COLUMNS = [
    "review_count",
    "low_rating_ratio",
    "past_3m_low_rating_ratio",
    "mean_helpful_vote",
    "verified_purchase_ratio",
    "reliability_ci_width",
]
TARGET = "is_low_rating_surge"

## 4. 시간순 학습/검증/테스트 분할

랜덤 split 금지 — 미래 달 데이터가 섞이면 데이터 누수 발생.

In [ ]:
train_df = df[df["year_month"] < "2022-01"]
valid_df = df[(df["year_month"] >= "2022-01") & (df["year_month"] < "2022-09")]
test_df  = df[df["year_month"] >= "2022-09"]

X_train, y_train = train_df[FEATURE_COLUMNS], train_df[TARGET]
X_valid, y_valid = valid_df[FEATURE_COLUMNS], valid_df[TARGET]
X_test,  y_test  = test_df[FEATURE_COLUMNS],  test_df[TARGET]

print(f"train {len(train_df)} / valid {len(valid_df)} / test {len(test_df)}")
print(f"label ratio  train {y_train.mean():.2%} / valid {y_valid.mean():.2%} / test {y_test.mean():.2%}")

### 4-1. 피처 스케일링

fit은 X_train으로만 — valid/test는 transform만 (누수 방지).

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)
X_test_scaled  = scaler.transform(X_test)

## 5. 결측치 확인

In [ ]:
X_train.isna().sum()
X_valid.isna().sum()
X_test.isna().sum()

## 6. 베이스라인 모델 학습 (로지스틱회귀)

`class_weight="balanced"`로 클래스 불균형(양성 9.47%) 보정.

In [ ]:
model = LogisticRegression(class_weight="balanced", max_iter=1000)
model.fit(X_train_scaled, y_train)

## 7. 검증셋 평가

accuracy는 불균형 데이터에 부적절 — PR-AUC / ROC-AUC 사용.

In [ ]:
valid_proba = model.predict_proba(X_valid_scaled)[:, 1]

print(f"ROC-AUC: {roc_auc_score(y_valid, valid_proba):.3f}")
print(f"PR-AUC : {average_precision_score(y_valid, valid_proba):.3f}")
print(classification_report(y_valid, model.predict(X_valid_scaled)))

## 8. 피처 중요도 (계수 해석)

In [ ]:
coef_df = pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "coefficient": model.coef_[0],
}).sort_values("coefficient", ascending=False)

coef_df

## 9. (비교군) LightGBM

다중공선성에 덜 민감한 트리 모델과 성능 비교.

In [ ]:
import lightgbm as lgb

lgb_model = lgb.LGBMClassifier(
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum()
)
lgb_model.fit(X_train, y_train)

lgb_proba = lgb_model.predict_proba(X_valid)[:, 1]
print(f"LightGBM ROC-AUC: {roc_auc_score(y_valid, lgb_proba):.3f}")
print(f"LightGBM PR-AUC : {average_precision_score(y_valid, lgb_proba):.3f}")

## 10. 최종 테스트셋 평가

검증셋으로 더 좋았던 모델을 골라 마지막에 한 번만 테스트셋에 적용.

In [ ]:
test_proba = model.predict_proba(X_test_scaled)[:, 1]

print(f"Test ROC-AUC: {roc_auc_score(y_test, test_proba):.3f}")
print(f"Test PR-AUC : {average_precision_score(y_test, test_proba):.3f}")

## 11. Walk-forward 반복검증

최종 test(2022-09~)는 재사용하지 않고, 그 이전 구간만 여러 번 잘라서 검증.

In [ ]:
def evaluate_fold(train_df, valid_df):
    X_tr, y_tr = train_df[FEATURE_COLUMNS], train_df[TARGET]
    X_va, y_va = valid_df[FEATURE_COLUMNS], valid_df[TARGET]

    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_va_s = scaler.transform(X_va)

    model = LogisticRegression(class_weight="balanced", max_iter=1000)
    model.fit(X_tr_s, y_tr)
    proba = model.predict_proba(X_va_s)[:, 1]

    return {
        "n_train": len(train_df),
        "n_valid": len(valid_df),
        "valid_label_ratio": y_va.mean(),
        "roc_auc": roc_auc_score(y_va, proba),
        "pr_auc": average_precision_score(y_va, proba),
    }

cutoff = df[df["year_month"] < "2022-09"]  # 최종 test 제외
months = sorted(cutoff["year_month"].unique())

MIN_TRAIN_MONTHS = 24  # 최소 2년 학습 후 검증 시작
VALID_WINDOW = 6       # 검증 구간 6개월씩 이동

results = []
start = MIN_TRAIN_MONTHS
while start + VALID_WINDOW <= len(months):
    train_end = months[start]
    valid_months = months[start:start + VALID_WINDOW]

    train_fold = cutoff[cutoff["year_month"] < train_end]
    valid_fold = cutoff[cutoff["year_month"].isin(valid_months)]

    m = evaluate_fold(train_fold, valid_fold)
    m["valid_start"], m["valid_end"] = valid_months[0], valid_months[-1]
    results.append(m)
    start += VALID_WINDOW

results_df = pd.DataFrame(results)
results_df

In [ ]:
# 요약: 구간별 성능이 얼마나 안정적인지
results_df[["roc_auc", "pr_auc"]].agg(["mean", "std"])